# Camada GOLD — Agregações Analíticas
Produz tabelas prontas para consumo por dashboards / relatórios.

In [1]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark('Gold - Agregação')
spark.sparkContext.setLogLevel('WARN')

SILVER = '/opt/spark/work-dir/warehouse/silver'
GOLD   = '/opt/spark/work-dir/warehouse/gold'

vendas   = spark.read.format('delta').load(f'{SILVER}/vendas')
clientes = spark.read.format('delta').load(f'{SILVER}/clientes')

26/06/18 16:36:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Receita mensal
receita_mensal = (
    vendas.groupBy('ano_mes')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .orderBy('ano_mes')
)
receita_mensal.show(15)
receita_mensal.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_mensal')

26/06/18 16:36:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+----------+--------------+------------+
|ano_mes|qtd_vendas| receita_total|ticket_medio|
+-------+----------+--------------+------------+
|2022-01|     20642| 2.774215023E8|    13439.66|
|2022-02|     18595|2.4804731275E8|    13339.46|
|2022-03|     20847|2.7716310211E8|    13295.11|
|2022-04|     20113| 2.657693231E8|    13213.81|
|2022-05|     21063|2.7965684995E8|    13277.16|
|2022-06|     20001|2.6875929658E8|    13437.29|
|2022-07|     20683|2.7267285414E8|    13183.43|
|2022-08|     20702|2.7388825742E8|    13230.04|
|2022-09|     19828|2.6253127853E8|    13240.43|
|2022-10|     20784|2.7549078683E8|    13254.95|
|2022-11|     20304|2.7043562656E8|    13319.33|
|2022-12|     20400|2.7090948706E8|    13279.88|
|2023-01|     20695|2.7555843549E8|    13315.22|
|2023-02|     18720|2.4902373696E8|    13302.55|
|2023-03|     20700|2.7445344966E8|    13258.62|
+-------+----------+--------------+------------+
only showing top 15 rows


In [3]:
# Receita por categoria
receita_categoria = (
    vendas.groupBy('categoria')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
    )
    .orderBy(F.desc('receita_total'))
)
receita_categoria.show()
receita_categoria.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_categoria')

+-----------+----------+---------------+
|  categoria|qtd_vendas|  receita_total|
+-----------+----------+---------------+
| Automotivo|     84472|1.18561356082E9|
|   Esportes|     78870|1.06247013104E9|
|Eletrônicos|     76779| 1.0416767035E9|
|  Alimentos|     78625| 9.9391114627E8|
|     Roupas|     77921| 9.8365242852E8|
|     Livros|     65966| 9.3380861786E8|
|     Beleza|     71178| 9.1972280368E8|
|       Casa|     66119| 8.5019907841E8|
+-----------+----------+---------------+



In [5]:
# Top 5 clientes por receita
clientes_nome = clientes.select(
    'cliente_id',
    F.col('nome').alias('nome_cliente')
)

top_clientes = (
    vendas.join(clientes_nome, 'cliente_id')
    .groupBy('cliente_id', 'nome_cliente')
    .agg(
        F.count('venda_id').alias('qtd_compras'),
        F.round(F.sum('valor_total'), 2).alias('total_gasto')
    )
    .orderBy(F.desc('total_gasto'))
    .limit(5)
)
top_clientes.show()
top_clientes.write.format('delta').mode('overwrite').save(f'{GOLD}/top_clientes')

+----------+------------------+-----------+-----------+
|cliente_id|      nome_cliente|qtd_compras|total_gasto|
+----------+------------------+-----------+-----------+
|     94428|       Brenda Lima|         16|  338332.24|
|     43112|       Yan Pimenta|         13|  326529.81|
|     44973|Ana Carolina Souza|         16|  326142.65|
|      7540|    Hellena Novais|         17|  322000.38|
|     64970|   Otávio Caldeira|         17|  313130.49|
+----------+------------------+-----------+-----------+



In [6]:
# Receita por região
receita_regiao = (
    vendas.groupBy('regiao')
    .agg(F.round(F.sum('valor_total'), 2).alias('receita_total'))
    .orderBy(F.desc('receita_total'))
)
receita_regiao.show()
receita_regiao.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_regiao')

+------------+---------------+
|      regiao|  receita_total|
+------------+---------------+
|Centro-Oeste|1.61734387411E9|
|       Norte| 1.6027906604E9|
|         Sul| 1.6014969038E9|
|    Nordeste|1.57601106435E9|
|     Sudeste|1.57341196744E9|
+------------+---------------+



In [7]:
# Histórico de versões de cada tabela Gold
from delta.tables import DeltaTable
for name in ['receita_mensal', 'receita_categoria', 'top_clientes', 'receita_regiao']:
    dt = DeltaTable.forPath(spark, f'{GOLD}/{name}')
    v  = dt.history(1).collect()[0]
    print(f'[gold/{name}] v{v["version"]} — {v["operation"]}')

[gold/receita_mensal] v0 — WRITE
[gold/receita_categoria] v0 — WRITE
[gold/top_clientes] v0 — WRITE
[gold/receita_regiao] v0 — WRITE
